In [ ]:
import os
import torch
import torch.nn as nn
import sys
sys.path.insert(0, r"C:\Not Windows\...")
import config

print("PyTorch version:", torch.__version__)
print("Device:", config.DEVICE)
print("Input size:", 126)
print("Output classes:", config.NUM_CLASSES)
print("Classes:", config.ASL_CLASSES)

PyTorch version: 2.5.1+cu121
Device: cuda
Input size: 126
Output classes: 29
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [2]:
class SignifyASLModel(nn.Module):
    def __init__(self, input_size=126, num_classes=29):
        super(SignifyASLModel, self).__init__()

        # Feature extraction layers
        self.features = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        features = self.features(x)
        output = self.classifier(features)
        return output

# Create model and move to GPU
model = SignifyASLModel(input_size=126, num_classes=config.NUM_CLASSES)
model = model.to(config.DEVICE)

# Print architecture summary
print("Signify ASL Model Architecture")
print("=" * 50)
print(model)
print("=" * 50)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model device: {next(model.parameters()).device}")

Signify ASL Model Architecture
SignifyASLModel(
  (features): Sequential(
    (0): Linear(in_features=126, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=29, bias=True)
  )
)
Total parameters:     241,181
Trainable parameters: 241,181
Model device: cuda:0


In [3]:
# Test model with a dummy batch before saving
batch_size = 32
dummy_input = torch.randn(batch_size, 126).to(config.DEVICE)

model.eval()
with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Expected:     ({batch_size}, {config.NUM_CLASSES})")
print(f"Forward pass: {'PASSED' if output.shape == (batch_size, config.NUM_CLASSES) else 'FAILED'}")

# Check output is reasonable
probs = torch.softmax(output, dim=1)
print(f"\nSample prediction probabilities sum: {probs[0].sum().item():.4f} (should be 1.0)")
print(f"Max probability in sample:          {probs[0].max().item():.4f}")
print(f"Predicted class index:              {probs[0].argmax().item()}")
print(f"Predicted class:                    {config.ASL_CLASSES[probs[0].argmax().item()]}")
print(f"\nModel ready for training")

Input shape:  torch.Size([32, 126])
Output shape: torch.Size([32, 29])
Expected:     (32, 29)
Forward pass: PASSED

Sample prediction probabilities sum: 1.0000 (should be 1.0)
Max probability in sample:          0.0378
Predicted class index:              11
Predicted class:                    L

Model ready for training


In [ ]:
import os

# Save model architecture to models/architecture/signify_model.py
arch_path = os.path.join(config.BASE_DIR, "models", "architecture", "signify_model.py")
os.makedirs(os.path.dirname(arch_path), exist_ok=True)

architecture_code = '''import torch
import torch.nn as nn

class SignifyASLModel(nn.Module):
    def __init__(self, input_size=126, num_classes=29):
        super(SignifyASLModel, self).__init__()

        self.features = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        features = self.features(x)
        output = self.classifier(features)
        return output

ASL_CLASSES = [
    "A","B","C","D","E","F","G","H","I","J","K","L","M",
    "N","O","P","Q","R","S","T","U","V","W","X","Y","Z",
    "del","nothing","space"
]
'''

with open(arch_path, 'w') as f:
    f.write(architecture_code)

print(f"Architecture saved to: {arch_path}")

Architecture saved to: C:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Capstone Project\models\architecture\cnn_lstm.py
Step 08 complete — ready for Step 09 training


In [3]:
class SignifyASLModelV2(nn.Module):
    def __init__(self, input_size=126, num_classes=29):
        super(SignifyASLModelV2, self).__init__()

        # Deeper feature extraction
        self.features = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        # Residual connection
        self.residual = nn.Linear(input_size, 128)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        features = self.features(x)
        # Add residual connection
        residual = self.residual(x)
        combined = features + residual
        output = self.classifier(combined)
        return output

# Test new model
model_v2 = SignifyASLModelV2(input_size=126, num_classes=29).to(config.DEVICE)
total_params = sum(p.numel() for p in model_v2.parameters())

# Quick forward pass test
dummy = torch.randn(32, 126).to(config.DEVICE)
with torch.no_grad():
    out = model_v2(dummy)

print("Model V2 architecture:")
print(model_v2)
print(f"\nTotal parameters: {total_params:,}")
print(f"Output shape: {out.shape}")
print(f"Forward pass: PASSED")

Model V2 architecture:
SignifyASLModelV2(
  (features): Sequential(
    (0): Linear(in_features=126, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=512, out_features=256, bias=True)
    (9): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=256, out_features=128, bias=True)
    (13): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.2, inplace=False)
  )
  (residual): Linear(in_features=126, out_features=128, bias=True)
  (classifier): Sequenti